## 1. Ი𐑼 Instalación de Dependencias
Instalación de paquetes externos requeridos para la manipulación de documentos en Python.

In [ ]:
!pip install pypdf python-docx openpyxl pandas -q

## 2. Ი𐑼 Importacion de modulos
Importación de librerías nativas y externas para la lógica del procesamiento.

In [ ]:
import gc
import json
import logging
import os
from pathlib import Path
import warnings

import docx
import pandas as pd
import pypdf

import re

## 3. Ი𐑼 Silenciado de logs
Silenciar advertencias generales y de pypdf

In [ ]:
for logger_name in ["pypdf", "pypdf._cmap", "pypdf._reader"]:
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.CRITICAL)
    logger.propagate = False
warnings.filterwarnings("ignore")

## 4. Ი𐑼 Conexión a Drive
Conexión con el entorno de Google Colab y montaje de Google Dirve

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Ი𐑼 Funciones de Extracción



*   leer_txt(): Extrae texto plano manejando codificación UTF-8.
*   leer_pdf(): Revisa y extrae el texto página por página usando pypdf.
*   leer_word(): Extrae párrafos de archivos .docx omitiendo líneas vacías.
*   leer_excel(): Recorre todas las pestañas de una hoja de cálculo y las convierte en texto estructurado.










*Ი𐑼 Lectura de archivos .txt*

In [ ]:
def leer_txt(ruta_archivo):
    """Abre y lee archivos .TXT con codificación UTF-8."""
    try:
        with open(ruta_archivo, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception as e:
        return f"[Error leyendo TXT: {str(e)}]"

*Ი𐑼 Lectura de archivos .pdf*

In [ ]:
def leer_pdf(ruta_archivo):
    """Extrae texto de archivos .PDF manejando excepciones por archivo."""
    texto_paginas = []
    try:
        lector = pypdf.PdfReader(ruta_archivo)
        for pagina in lector.pages:
            try:
                contenido = pagina.extract_text()
                if contenido:
                    texto_paginas.append(contenido)
            except Exception:
                continue
    except Exception as e:
        return f"[Error leyendo PDF: {str(e)}]"

    return "\n".join(texto_paginas)

*Ი𐑼 Lectura de archivos .docx*

In [ ]:
def leer_word(ruta_archivo):
    """Extrae texto de archivos .DOCX omitiendo líneas vacías."""
    try:
        doc = docx.Document(ruta_archivo)
        return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        return f"[Error leyendo DOCX: {str(e)}]"

*Ი𐑼 Lectura de archivos .xlsx*

In [ ]:
def leer_excel(ruta_archivo):
    """Extrae y formatea el contenido de archivos .XLSX/XLS."""
    try:
        dict_hojas = pd.read_excel(ruta_archivo, sheet_name=None)
        texto_completo = []
        for nombre_hoja, df in dict_hojas.items():
            texto_completo.append(f"--- Hoja: {nombre_hoja} ---")
            texto_completo.append(df.to_string(index=False))
        return "\n".join(texto_completo)
    except Exception as e:
        return f"[Error leyendo Excel: {str(e)}]"

## 6. Ი𐑼 (pipeline_procesar_documento)



1.   Valida la existencia del archivo en el sistema.
2.   Identifica la extensión del documento.
1.   Ejecuta la función de extracción correspondiente.
2.   Normaliza y limpia los espacios en blanco sobrantes.
1.   Retorna un diccionario estructurado con los metadatos y el contenido.


*Ი𐑼 Pipeline*

In [ ]:
def pipeline_procesar_documento(ruta_archivo):
    """Identifica la extensión y ejecuta la lectura correspondiente."""
    path = Path(ruta_archivo)
    ext = path.suffix.lower()

    if ext == ".txt":
        contenido = leer_txt(ruta_archivo)
    elif ext == ".pdf":
        contenido = leer_pdf(ruta_archivo)
    elif ext == ".docx":
        contenido = leer_word(ruta_archivo)
    elif ext in [".xlsx", ".xls"]:
        contenido = leer_excel(ruta_archivo)
    else:
        return None

    return {
        "nombre_archivo": path.name,
        "extension": ext,
        "contenido": contenido
    }

def procesar_carpeta_a_json(ruta_carpeta, ruta_salida_json="resultado_documentos.json"):
    """
    Recorre recursivamente la carpeta, gestiona la memoria de la máquina
    y exporta la lista de documentos a JSON.
    """
    carpeta = Path(ruta_carpeta)
    extensiones_validas = {".pdf", ".docx", ".txt", ".xlsx", ".xls"}
    resultados = []

    if not carpeta.exists():
        print(f"✘ Error: La ruta '{ruta_carpeta}' no existe.")
        return

    print(f"𓄲 Escaneando carpeta: {carpeta.name}...\n")

    archivos = [a for a in carpeta.rglob("*") if a.is_file() and a.suffix.lower() in extensiones_validas]
    total_archivos = len(archivos)

    for i, archivo in enumerate(archivos, 1):
        print(f"⌨ [{i}/{total_archivos}] Procesando: {archivo.name}")

        try:
            datos = pipeline_procesar_documento(str(archivo))
            if datos:
                datos["ruta"] = str(archivo)
                resultados.append(datos)
        except Exception as e:
            resultados.append({
                "nombre_archivo": archivo.name,
                "ruta": str(archivo),
                "error": str(e)
            })

        # Liberación forzada de memoria RAM cada 20 archivos
        if i % 20 == 0:
            gc.collect()

    # Guardar archivo JSON estructurado (indentación 2 para reducir peso de archivo)
    with open(ruta_salida_json, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

    gc.collect()

    print(f"\n✔ ¡Proceso completado con éxito!")
    print(f"〽︎ Total de documentos procesados: {len(resultados)}")
    print(f"⎙ Archivo generado: {ruta_salida_json}")

## 7. Ი𐑼 Limpieza de Json

In [ ]:
def limpiar_texto_basico(texto: str) -> str:
    """Elimina basura de formato en el texto crudo extraído."""
    if not texto or texto.startswith("[Error"):
        return ""

    texto = re.sub(r"\n+", "\n", texto)
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"[\r\t\f\v]", "", texto)
    lineas = [linea.strip() for linea in texto.split("\n") if linea.strip()]

    return "\n".join(lineas).strip()


def procesar_limpieza_json(
    ruta_json_entrada, ruta_json_salida="resultado_documentos_limpio.json"
):
    """Toma el JSON extraído, limpia cada texto y guarda el JSON final."""
    if not os.path.exists(ruta_json_entrada):
        print(f"✘ Error: No existe el archivo '{ruta_json_entrada}'")
        return

    with open(ruta_json_entrada, "r", encoding="utf-8") as f:
        documentos_raw = json.load(f)

    print(f"𓄲 Limpiando texto de {len(documentos_raw)} documentos...")

    documentos_limpios = []
    for doc in documentos_raw:
        texto_limpio = limpiar_texto_basico(doc.get("contenido", ""))
        if texto_limpio:
            doc_nuevo = doc.copy()
            doc_nuevo["contenido"] = texto_limpio
            documentos_limpios.append(doc_nuevo)

    with open(ruta_json_salida, "w", encoding="utf-8") as f:
        json.dump(documentos_limpios, f, ensure_ascii=False, indent=2)

    print(
        f"✔ ¡Limpieza finalizada! Guardados {len(documentos_limpios)} documentos en '{ruta_json_salida}'."
    )

## 8. Ი𐑼 Prueba

In [ ]:
# Definir la ruta de la carpeta en Google Drive
ruta_drive = "/content/drive/MyDrive/0 Legislación SSO-20260724T184454Z-1-001/0 Legislación SSO"

# Extracción
print("➔ PASO 1: Extrayendo documentos...")
procesar_carpeta_a_json(
    ruta_carpeta=ruta_drive, ruta_salida_json="resultado_documentos.json"
)

# Limpieza
print("\n➔ PASO 2: Limpiando data...")
procesar_limpieza_json(
    ruta_json_entrada="resultado_documentos.json",
    ruta_json_salida="resultado_documentos_limpio.json",
)